In [11]:
import pandas as pd
import numpy as np 
covid=pd.read_csv("z.Data/covid_data.csv") 
covid.head()

,Date,Country,Total_Cases,New_Cases,Total_Deaths,New_Deaths,Recovered,Active_Cases,Serious_Critical,Tests_Per_Million,Population,Alert
0,2023-01-01,USA,99732855,14528,1073788,122,95683961,4847881,8626,1764678,338290868,Red
1,2023-01-08,India,47028214,8034,516590,51,45756120,668220,2128,339037,1405845055,Red
2,2023-01-15,Brazil,37858378,7261,723235,44,36526173,729996,3215,165300,204389564,Red
3,2023-01-22,France,38374902,12279,154043,86,36202125,336528,1820,215602,70248138,Yellow
4,2023-01-29,Germany,34761667,11337,158306,79,34254836,350494,1642,192088,83769689,Yellow


In [12]:
from sklearn.model_selection import train_test_split
x=covid.drop(columns=["Date", "Country", "New_Deaths"] )
y=covid["New_Deaths"] 

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
covid.shape

(1000, 12)

In [3]:
num_cols = [
    'Total_Cases',
    'New_Cases',    
    'Total_Deaths',
    'Recovered',
    'Active_Cases',
    'Serious_Critical',
    'Tests_Per_Million',
    'Population',
] 
cat=['Alert']

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

preprocessor = ColumnTransformer(transformers=[ 
    ('num', Pipeline(steps=[ 
        ('imputer', SimpleImputer(strategy='median')), 
        ('scaler', StandardScaler())                  
    ]), num_cols), 
    
    ('cat', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')), 
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
         #- Converts categorical variables into binary (0/1) columns.
    ]), cat)
])

In [6]:
from sklearn.ensemble import RandomForestRegressor
model = Pipeline(steps=[
    ('prep', preprocessor), 
    ('regressor', RandomForestRegressor(n_estimators=500, random_state=42))
])
model.fit(X_train, y_train) 
score = model.score(X_test, y_test)
score

0.9970218670657506

In [8]:
from sklearn.model_selection import cross_val_score
cross= cross_val_score(model, x,y,cv=5)
cross, cross.mean() 

(array([0.99834083, 0.99872783, 0.99813097, 0.98300591, 0.9969562 ]),
 np.float64(0.995032346029231))